In [1]:
import os

# --- JAX / CUDA GPU MEMORY MANAGEMENT CONFIGURATION ---
# MUST be set at the very top of the notebook BEFORE 'import jax' or 'import flax'!
# 1. Disable JAX's default 75% GPU memory pre-allocation.
#    By default, JAX claims 75% of total GPU VRAM upfront upon initialization.
#    Setting this to "false" forces JAX to allocate GPU memory dynamically as needed,
#    preventing Out-Of-Memory (OOM) errors when sharing GPUs or running large intermediate graphs.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"] = "0.9"
# 2. Use the CUDA Async Memory Allocator for XLA.
#    Replaces XLA's default rigid BFC (Best-Fit with Coalscing) allocator with CUDA's native 
#    async pool allocator. This reduces memory fragmentation during large intermediate tensor 
#    transformations (e.g. high-resolution L=1500 spherical convolutions/FFTs).
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"

print("--- Environment Variables inside Container ---")
print("CUDA_VISIBLE_DEVICES  :", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("NVIDIA_VISIBLE_DEVICES:", os.environ.get("NVIDIA_VISIBLE_DEVICES"))

print("\n--- Testing nvidia-smi ---")
os.system("nvidia-smi")

import socket

print("=== VS CODE CONTAINER DIAGNOSTIC ===")
print("1. Hostname               :", socket.gethostname())
print("2. Slurm Job ID           :", os.environ.get("SLURM_JOB_ID", "NONE (Running outside Slurm!)"))
print("3. CUDA_VISIBLE_DEVICES   :", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("4. Apptainer GPU Devices  :", [f for f in os.listdir('/dev') if 'nvidia' in f] if os.path.exists('/dev') else "No /dev")

import sys

print("1. Python Binary  :", sys.executable)
print("2. Site-Packages  :", [p for p in sys.path if ".venv" in p])
print("3. CUDA Available :", end=" ")

import torch
print(torch.cuda.is_available(), f"({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "")

--- Environment Variables inside Container ---
CUDA_VISIBLE_DEVICES  : 0,1
NVIDIA_VISIBLE_DEVICES: all

--- Testing nvidia-smi ---
Wed Jul 29 12:00:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 565.57.01              Driver Version: 565.57.01      CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GH200 120GB             On  |   00000019:01:00.0 Off |                    0 |
| N/A   33C    P0            100W /  900W |       1MiB /  97871MiB |      0%      Default |
|        

/opt/venv/lib/python3.13/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12070). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
import os
# 1. Force JAX to see the GPU
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# 2. Keep CPU threads from crashing if it falls back
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["MKL_NUM_THREADS"] = "4"
os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=1"
# 3. Provide valid temporary directory for ptxas CUDA kernel compilation
os.environ["TMPDIR"] = "/tmp"
os.environ["TMP"] = "/tmp"
os.environ["TEMP"] = "/tmp"

import dataclasses
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
print("Jax version", jax.__version__, "64-bit", jax.config.read("jax_enable_x64"))
import jax.numpy as jnp
import flax
print("Flax version", flax.__version__)
from flax import nnx
import optax
# from sbi_compression.methods.utils import universal_flax_nnx_shim
# universal_flax_nnx_shim()

from torch.utils.data import Dataset, DataLoader, random_split

from s2ai.blocks.core_blocks import DiscoConvBlock, AverageBlock
from typing import Any, List, Optional, Callable, Sequence, Tuple, Union
from jaxtyping import Array, Float, Int, PyTree # https://github.com/google/jaxtyping

from typing import Literal, Callable, Optional
from tqdm import tqdm
from math import prod


class DiscoCNN(nnx.Module):
    """Disco convolutional network with spherical group normalisation."""

    def __init__(
        self,
        modes: List[str] = dataclasses.field(
            default_factory=lambda: ["disco", "disco", "disco", "disco"]
        ),
        rngs: nnx.Rngs = nnx.Rngs(0),
    ):
        # These could be nnx.vmapped/scanned for faster initialization
        self.disco_conv_block_0 = DiscoConvBlock(
            (128, 64), (1, 4), 2, mode=modes[0], rngs=rngs
        )
        self.disco_conv_block_1 = DiscoConvBlock(
            (64, 32), (4, 16), 4, mode=modes[1], rngs=rngs
        )
        self.disco_conv_block_2 = DiscoConvBlock(
            (32, 16), (16, 64), 8, mode=modes[2], rngs=rngs
        )
        self.disco_conv_block_3 = DiscoConvBlock(
            (16, 8), (64, 256), 16, mode=modes[3], rngs=rngs
        )

        self.average_block = AverageBlock(L_in=8)

        self.dropout_4 = nnx.Dropout(0.5, rngs=rngs)
        self.dense_4 = nnx.Linear(
            in_features=256,
            out_features=256,
            rngs=rngs,
        )
        self.activation_4 = nnx.swish

        self.dropout_5 = nnx.Dropout(0.5, rngs=rngs)
        self.dense_5 = nnx.Linear(
            in_features=256,
            out_features=10,
            rngs=rngs,
        )
        self.activation_5 = nnx.log_softmax

    def __call__(self, x):
        # Spherical convolutional blocks (L's, channels, groups)
        x = self.disco_conv_block_0(x)
        x = self.disco_conv_block_1(x)
        x = self.disco_conv_block_2(x)
        x = self.disco_conv_block_3(x)

        # Convert to equivariant dense layer
        x = self.average_block(x)

        # Standard MLP classifier
        x = self.dropout_4(x)
        x = self.dense_4(x)
        x = self.activation_4(x)
        x = self.dropout_5(x)
        x = self.dense_5(x)
        x = self.activation_5(x)
        return x



Jax version 0.11.0 64-bit True
Flax version 0.12.7


E0729 12:00:34.352038  218500 numa_hwloc.cc:121] Call to hwloc_set_cpubind() failed: Invalid argument [22]


In [3]:
class DiscoCNNBuild(nnx.Module):
    "Builds spherical convolutional network based on set of tuples defining the convolution block per layer."
    def __init__(
        self,
        input_shape: tuple[int, ...],
        output_dim: int,
        layer_dims: list,
        activation: Callable = nnx.leaky_relu,
        rngs: nnx.Rngs | None = None, 
        ):

        if rngs is None:
            rngs = nnx.Rngs(0)

        self.input_shape = tuple(input_shape)
        self.output_dim = output_dim
        self.layer_dims = tuple(tuple(x) if not isinstance(x, tuple) else x for x in layer_dims)
        if len(self.layer_dims) == 0:
            raise ValueError("At least one layer dimension must be specified in layer_dims.")
        self.activation = activation
        self.rngs = rngs
        self.layers = nnx.List()
        
        raise NotImplementedError

    def check_layer_dims(
        self,
        input_shape: tuple[int, ...],
        output_dim: int,
        layer_dims: list,
        ):
        
        raise NotImplementedError

    def _add_layer(
        self,
        layer_dim: tuple,
        prev_layer_dim: tuple | None,
        layers: nnx.List,
        ):

        if prev_layer_dim is None:
            if len(layer_dim) == 2:
                if len(self.input_shape) == 1:
                    if self.input_shape[0] != layer_dim[0]:
                        raise ValueError(
                            f"Expected input shape {self.input_shape} to match linear "
                            f"input dimension {layer_dim[0]}."
                        )
                    layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))
                elif len(self.input_shape) == 3:
                    if prod(self.input_shape) != layer_dim[0]:
                        raise ValueError(
                            f"Expected flattened input size {prod(self.input_shape)} to "
                            f"match linear input dimension {layer_dim[0]}."
                        )
                    layers.append(self.Flatten())
                    layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))
                else:
                    raise ValueError(f"Unsupported input shape {self.input_shape}.")

            elif len(layer_dim) == 3:
                if len(self.input_shape) != 3:
                    raise ValueError(
                        f"Cannot add a convolutional layer after a linear layer, got "
                        f"layer dimensions {layer_dim} after input shape {self.input_shape}."
                    )
                if self.input_shape[2] != layer_dim[0]:
                    raise ValueError(
                        f"Expected input channels {self.input_shape[2]} to match "
                        f"convolutional input channels {layer_dim[0]}."
                    )
                layers.append(
                    nnx.Conv(
                        layer_dim[0],
                        layer_dim[1],
                        kernel_size=(layer_dim[2], layer_dim[2]),
                        padding="VALID",
                        rngs=self.rngs,
                    )
                )   

        else:
            if len(layer_dim) == 2 and len(prev_layer_dim) == 2:
                if prev_layer_dim[1] != layer_dim[0]:
                    raise ValueError(
                        f"Linear layer input dimension {layer_dim[0]} does not match "
                        f"previous layer output dimension {prev_layer_dim[1]}."
                    )
                layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))

            elif len(layer_dim) == 3 and len(prev_layer_dim) == 3:
                if prev_layer_dim[1] != layer_dim[0]:
                    raise ValueError(
                        f"Convolutional layer input channels {layer_dim[0]} do not match "
                        f"previous layer output channels {prev_layer_dim[1]}."
                    )
                layers.append(
                    nnx.Conv(
                        layer_dim[0],
                        layer_dim[1],
                        kernel_size=(layer_dim[2], layer_dim[2]),
                        padding="VALID",
                        rngs=self.rngs,
                    )
                )

            elif len(layer_dim) == 2 and len(prev_layer_dim) == 3:
                layers.append(self.Flatten())
                layers.append(nnx.Linear(layer_dim[0], layer_dim[1], rngs=self.rngs))

            elif len(layer_dim) == 3 and len(prev_layer_dim) == 2:
                raise ValueError(
                    f"Cannot add a convolutional layer after a linear layer, got "
                    f"layer dimensions {layer_dim} after {prev_layer_dim}."
                )

        raise NotImplementedError
    
    def __call__(self, x):
        if len(self.layers) > 1:
            for layer in self.layers[:-1]:
                x = layer(x)
                x = self.activation(x)
            x = self.layers[-1](x)
        elif len(self.layers) == 1:
            for layer in self.layers:
                x = layer(x)
        return x


In [4]:
# directory = "/Users/brian/Downloads/map_compression_samples_5_tomo_bins_flm_1500_batch_1"
directory = "/scratch/u6pf/brianycc.u6pf/spherical_maps/map_compression_samples_5_tomo_bins_flm_1500_batch_1/"
# n_map = 1
t = np.load(directory+f'/map_{1}.npy')
print(t.shape)
p = np.load(directory+"/params.npy")
print(p.shape)

# m = []
# for i in range(n_map):
#     m_add = np.load(directory+f'/map_{i}.npy')
#     m.append(m_add)
#     if i % 50 == 0 :
#         print(m_add.shape)
#         print(len(m))
# m = np.asarray(m)
# m = m.reshape((-1, 1500, 2*1500-1, 5)) 

(22492500,)
(2000, 5)


In [ ]:
import s2fft

class SphericalMapDataset(Dataset):
    def __init__(self, directory, n_maps=1000, param_filename="params.npy"):
        """
        Args:
            directory (str): Path to the folder containing map_{i}.npy.
            n_maps (int): Total number of map samples to load.
            param_filename (str): Name of the parameters numpy file.
        """
        self.directory = directory
        self.n_maps = n_maps
        
        # The parameters file is tiny, so it is safe to load entirely into RAM
        param_path = os.path.join(directory, param_filename)
        self.params = np.load(param_path)
        
        assert self.n_maps <= len(self.params), \
            f"Requested {self.n_maps} maps, but params.npy only has {len(self.params)} rows!"
        
    def __len__(self):
        return self.n_maps
    
    def __getitem__(self, idx):
        # 1. Lazily load only ONE map on demand
        map_path = os.path.join(self.directory, f"map_{idx}.npy")
        m = np.load(map_path)
        
        # 2. Reshape it appropriately
        m = m.reshape((1500, 2999, 5))
        
        # 3. Best practice: cast to float32 for Neural Network training
        m = m.astype(np.float32)
        # Convert MW → MWSS for each tomographic channel
        # MWSS(L=1500): (1501, 3000) theta x phi
        m_mwss = np.stack([
            np.array(s2fft.utils.resampling.mw_to_mwss(m[..., c], L=1500)).real
            for c in range(5)
            ], axis=-1)  # shape: (1501, 3000, 5)
        m_750 = np.stack([
            np.array(s2fft.utils.resampling.downsample_by_two_mwss(m_mwss[..., c], L=750)).real
            for c in range(5)
            ], axis=-1)  # shape: (1501, 3000, 5)
        # m_750 = s2fft.utils.resampling.downsample_by_two_mwss(m_mwss, L=750)

        # 4. Get corresponding parameter vector
        p = self.params[idx].astype(np.float32)
        
        # PyTorch DataLoaders will automatically stack these into tensors
        return m_750, p


In [6]:
class In1500_Out5_DiscoCNN(nnx.Module):
    """Disco convolutional network with spherical group normalisation."""

    def __init__(
        self,
        # modes: List[str] = dataclasses.field(
        #     default_factory=lambda: ["disco", "disco", "disco", "disco"]
        # ),
        modes: tuple[str] = ("disco", "disco", "disco", "disco"),
        rngs: nnx.Rngs = nnx.Rngs(0),
    ):
        # These could be nnx.vmapped/scanned for faster initialization
        # self.disco_conv_block_0 = DiscoConvBlock(
        #     (1500, 750), (5, 5), 5, mode=modes[0], rngs=rngs
        # )
        self.disco_conv_block_1 = DiscoConvBlock(
            (750, 250), (5, 10), 10, mode=modes[1], rngs=rngs
        )
        self.disco_conv_block_2 = DiscoConvBlock(
            (250, 50), (10, 20), 20, mode=modes[2], rngs=rngs
        )
        self.disco_conv_block_3 = DiscoConvBlock(
            (50, 10), (20, 40), 40, mode=modes[3], rngs=rngs
        )

        self.average_block = AverageBlock(L_in=10)

        self.dropout_4 = nnx.Dropout(0.5, rngs=rngs)
        self.dense_4 = nnx.Linear(
            in_features=40,
            out_features=20,
            rngs=rngs,
        )
        self.activation_4 = nnx.swish

        self.dropout_5 = nnx.Dropout(0.5, rngs=rngs)
        self.dense_5 = nnx.Linear(
            in_features=20,
            out_features=5,
            rngs=rngs,
        )
        self.activation_5 = nnx.log_softmax

    @nnx.remat
    def __call__(self, x):
        # Spherical convolutional blocks (L's, channels, groups)
        # x = self.disco_conv_block_0(x)
        x = self.disco_conv_block_1(x)
        x = self.disco_conv_block_2(x)
        x = self.disco_conv_block_3(x)
        jax.debug.print("Past disco block.")
        # Convert to equivariant dense layer
        x = self.average_block(x)
        # Standard MLP classifier
        x = self.dropout_4(x)
        x = self.dense_4(x)
        x = self.activation_4(x)
        x = self.dropout_5(x)
        x = self.dense_5(x)
        # x = self.activation_5(x)
        jax.debug.print("Past dense mlp.")
        return x

s2_CNN = In1500_Out5_DiscoCNN()

from sbi_compression.methods.neural.flows import RQSplineFlow
import distrax

features_shape = (5,)
context_shape = (5,)
features_dim = prod(features_shape)
context_dim = prod(context_shape)
n_transforms = 4
n_bins = 8
range_min = -5
range_max = 5
bijector_type = distrax.RationalQuadraticSpline
conditioner_hidden_dims = ((32,32), (32,32))
activation = "gelu"
flow = RQSplineFlow(
    features_dim, 
    context_dim, 
    n_transforms=n_transforms, 
    hidden_dims=conditioner_hidden_dims, 
    activation=getattr(nnx, activation), 
    n_bins=n_bins, 
    range_min=range_min, 
    range_max=range_max, 
    bijector_type=bijector_type
    )

class s2_AE_Flow(nnx.Module):

    def __init__(self,
        encoder: nnx.Module,
        flow: nnx.Module,
        encoder_mode: Literal['context', 'features'],
        ):
        self.encoder_mode = encoder_mode
        self.encoder = encoder
        self.flow = flow
    
    def __call__(self, x: Array, context: Array) -> Array:
        if self.encoder_mode == 'context':
            context_encoded = self.encoder(context)
            x = self.flow(x, context_encoded)
        elif self.encoder_mode == 'features':
            x_encoded = self.encoder(x)
            x = self.flow(x_encoded, context)
        print("Past flow.")
        return x
    
    def sample(self, num_samples: int, rng: Array, context: Array) -> Array:
        x = self.flow.sample(num_samples, rng, context)
        return x
        
    def encode(self, x: Array) -> Array:
        # assert x.shape[-len(self.input_shape):] == self.input_shape # Make sure the input shape matches the encoder, assume batch first
        x = self.encoder(x)
        return x
    
    def mode(self, mode: Literal['train_encoder', 'train_flow', 'train_all', 'eval']) -> None:
        if mode == 'train_encoder':
            self.encoder.train()
            self.flow.eval()
        elif mode == 'train_flow':
            self.encoder.eval()
            self.flow.train()
        elif mode == 'train_all':
            self.encoder.train()
            self.flow.train()
        elif mode == 'eval':
            self.encoder.eval()
            self.flow.eval()
    
S2_VMIM = s2_AE_Flow(s2_CNN, flow, 'context')


In [ ]:
x_batch, p_batch = next(infinite_trainloader())
print(x_batch.shape)
print(p_batch.shape)

: 

In [ ]:
LEARNING_RATE = 5e-5
TRAIN_TEST_SPLIT = 0.8
BATCH_SIZE = 64
STEPS = 10
PRINT_EVERY = 2

# Loss function
# def mse_loss(model, x, y):
#     preds = model(x, y)
#     if preds.shape != y.shape:
#         raise ValueError(f"Output shpae of the model {preds.shape} does not match the shape of the labels {y.shape}")
#     return jnp.mean((preds-y) ** 2)
def nll_loss(model, x, context):
    loss = -jnp.mean(model(x, context))
    return loss

# Optimiser
optimizer = nnx.Optimizer(
    S2_VMIM, 
    optax.adamw(LEARNING_RATE),
    wrt=nnx.Param
)

@nnx.jit(static_argnames="loss_fn")
def train_step(model, optimizer: nnx.Optimizer, loss_fn, x_batch, context_batch):
    """Train for a single step."""
    loss_value, grads = nnx.value_and_grad(loss_fn)(model, x_batch, context_batch)
    optimizer.update(model, grads)  # In-place updates.
    return loss_value

@nnx.jit(static_argnames="loss_fn")
def eval_step(model, loss_fn, x, context):
    """Calculate loss on test data without updating parameters."""
    loss_value = loss_fn(model, x, context)
    return loss_value

def infinite_trainloader():
    while True:
        yield from train_loader

train_losses = []
test_losses = []
test_steps = []

dataset = SphericalMapDataset(
    directory="/projects/u6pf/brianycc/spherical_maps/map_compression_samples_5_tomo_bins_flm_1500_batch_1/", 
    n_maps=1000, 
    param_filename="params.npy"
    )
train_size = int(TRAIN_TEST_SPLIT * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, num_workers=4, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=True, num_workers=4, persistent_workers=True)


# for step, (x_batch, p_batch) in tqdm(zip(range(STEPS), infinite_trainloader())):
#     train_loss = train_step(S2_VMIM, optimizer, nll_loss, p_batch.numpy(), x_batch.numpy()) # Posteriior Estimation
#     # Block until GPU finishes step (prevents asynchronous memory buildup)
#     train_loss.block_until_ready()
#     train_losses.append(float(train_loss))
#     # Explicitly delete batch references so Python garbage collector frees host RAM
#     del x_batch, p_batch
#     # --- EVALUATION PHASE ---
#     if step % PRINT_EVERY == 0:
#         # metrics.reset() # Clear training metrics to track test metrics
#         test_loss = 0
#         for batch_x, batch_p in test_loader:
#             test_loss += eval_step(S2_VMIM, nll_loss, batch_p.numpy(), batch_x.numpy())
#         test_loss /= len(test_loader)
#         test_loss.block_until_ready()
#         test_losses.append(float(train_loss))
#         del batch_x, batch_p
#         test_steps.append(step)
#         print(f"Step {step:3d} ({(step*BATCH_SIZE)/train_size:.1f} epoch) | Train Loss: {train_loss:.6f} | Test Loss: {test_loss:.6f}")
# print("Training completed.")


# import pickle

# # --- SAVING ---
# # 1. Get the model state (parameters, batch stats, etc.)
# state = nnx.state(s2_VMIM, nnx.Param)

# # 2. Save it to a file
# with open('checkpoints/s2VMIM_v1.pkl', 'wb') as f:
#     pickle.dump(state, f)



In [ ]:
import pickle

# --- SAVING ---
# 1. Get the model state (parameters, batch stats, etc.)
state = nnx.state(, nnx.Param)

# 2. Save it to a file
with open('checkpoints/autoencoder_flow_v1.pkl', 'wb') as f:
    pickle.dump(state, f)

# --- LOADING ---
# 1. Create a fresh instance of the model with the SAME hyperparameters
new_model = AutoencoderFlow(
    input_shape = (60,60,5),
    features_shape = (n_features,),
    context_shape = (n_context,),
    encoder_layer_dims = ((5,3,11), (3,1,11), (1,1,11), (30*30, 100), (100, n_features)),
    conditioner_hidden_dims = ((32,32), (32,32)),
    encoder_mode = 'context',
    activation = 'gelu',
    n_transforms = 4,
    n_bins = 10,
    range_min = -4.0,
    range_max = 4.0,
    bijector_type = distrax.RationalQuadraticSpline
)

# 2. Load the state from the file
with open('checkpoints/autoencoder_flow_v1.pkl', 'rb') as f:
    loaded_state = pickle.load(f)

# 3. Update the model with the loaded state
nnx.update(new_model, loaded_state)

In [ ]:
# Training loss time series
fig, ax = plt.subplots()
for r,l,c in zip(results_files,labels,colours):
    with open("compressed_data/"+r, 'rb') as f:
        load = pickle.load(f)
        train_lossses = load["train_losses"]
        test_losses = load["test_losses"]
        test_steps = load["test_steps"]
        ax.plot(range(STEPS), train_losses, color=c, alpha=0.1)
        ax.plot(test_steps, test_losses, label=l, color=c)
        ax.set_xlim(0,STEPS)
plt.grid()
plt.legend()
plt.xlabel("Steps")
plt.ylabel("loss (expected log_p)")
plt.title(f"VMIM compression")
plt.show()